# Tugas 3 | Web Crawler & Ekstraksi Putusan Narkotika [SUBCPMK-3]
### Mata Kuliah: Temu Kembali Informasi (TKI)

---

**Identitas Kelompok:**
- **Nama Anggota 1:** Akhmad Zamri Ardani (NIM: 202310370311406)
- **Nama Anggota 2:** M Rifqi Dzaky Arrafi (NIM: 202310370311441)
- **Program Studi:** Informatika, Fakultas Teknik, Universitas Muhammadiyah Malang
- **Klasifikasi Perkara:** Pidana Khusus — Narkotika dan Psikotropika (Tahun 2023)
- **Lembaga Target:** Pengadilan Negeri Jember (`pn-jember`)
- **Sumber Data:** [putusan3.mahkamahagung.go.id](https://putusan3.mahkamahagung.go.id)

---

### Alur Pipeline Otomasi:
```
[1] Setup Lingkungan  --> [2] Konfigurasi Folder & Drive --> [3] Crawl Link Putusan
                      --> [4] Unduh Salinan PDF         --> [5] Konversi PDF ke TXT
                      --> [6] Ekstraksi Regex ke CSV    --> [7] Audit & Validasi Dataset
```

> **Catatan Eksekusi Google Colab:**
> - Jika dijalankan di **Google Colab**, notebook ini secara otomatis menginstal dependensi browser headless Chromium, virtual display (`xvfb`), dan menghubungkan Google Drive agar berkas unduhan tersimpan permanen.
> - Jika dijalankan secara **Lokal** (Windows/Linux/macOS), script langsung menyimpan berkas di folder direktori kerja saat ini.


---
## 1. Setup Lingkungan, Instalasi Library & Konfigurasi Browser
Mempersiapkan dependensi Python (SeleniumBase, PyMuPDF, BeautifulSoup4, Pandas) serta menginstal xvfb untuk display virtual pada runtime Colab/Linux.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 1: Konfigurasi Dependensi & Lingkungan Eksekusi (Google Colab & Lokal)
import sys
import shutil
import subprocess
import platform
from pathlib import Path

# Identifikasi runtime eksekusi
ENV_IS_COLAB = 'google.colab' in sys.modules
ENV_IS_LINUX = platform.system() == 'Linux'

print(f"[STATUS] Platform: {platform.system()} | Google Colab: {ENV_IS_COLAB}")

# Daftar paket Python yang dibutuhkan
REQUIRED_PACKAGES = [
    ('seleniumbase', 'seleniumbase'),
    ('fitz', 'pymupdf'),
    ('bs4', 'beautifulsoup4'),
    ('pandas', 'pandas'),
    ('requests', 'requests'),
    ('openpyxl', 'openpyxl'),
]

for mod, pkg in REQUIRED_PACKAGES:
    try:
        __import__(mod)
        print(f"[DEP] {pkg} sudah terpasang.")
    except ImportError:
        print(f"[INSTALL] Memasang modul {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"[DEP] {pkg} berhasil dipasang.")

# Khusus Google Colab / Linux: pastikan Google Chrome Browser dan xvfb terpasang
if ENV_IS_COLAB or (ENV_IS_LINUX and shutil.which('google-chrome') is None):
    print("[COLAB] Memeriksa dan memasang Google Chrome browser & xvfb...")
    subprocess.run(['apt-get', 'update', '-qq'], stdout=subprocess.DEVNULL)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'xvfb', 'wget'], stdout=subprocess.DEVNULL)
    
    if shutil.which('google-chrome') is None and shutil.which('google-chrome-stable') is None:
        print("[INSTALL] Mengunduh dan memasang Google Chrome Stable...")
        cmd_install_chrome = (
            "wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && "
            "apt-get install -y -qq ./google-chrome-stable_current_amd64.deb && "
            "rm -f google-chrome-stable_current_amd64.deb"
        )
        subprocess.run(cmd_install_chrome, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
    try:
        subprocess.run(['sbase', 'install', 'chromedriver'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass
    print("[DEP] Google Chrome dan virtual display xvfb siap digunakan.")

print("[SELESAI] Seluruh dependensi berhasil disiapkan.")


---
## 2. Analisis Kebijakan Kesusilaan robots.txt & Etika Web Crawling [SUBCPMK-3]
Memeriksa berkas robots.txt resmi Mahkamah Agung RI untuk memastikan proses crawling mematuhi prinsip etika (crawling courtesy): kepatuhan crawl-delay 10 detik, jeda acak 8-12 detik, dan legalitas akses dokumen putusan publik.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 2: Inspeksi Kebijakan Kesusilaan robots.txt & Etika Crawling
import urllib.request
import re

ROBOTS_URL = "https://putusan3.mahkamahagung.go.id/robots.txt"

print("=" * 75)
print("AUDIT KEBIJAKAN KESUSILAAN ROBOTS.TXT (ETIKA CRAWLER MA RI)")
print("=" * 75)
print(f"[TARGET] Mengambil direktif dari: {ROBOTS_URL}")

try:
    req = urllib.request.Request(
        ROBOTS_URL,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) TKI-StudentCrawler/1.0'}
    )
    with urllib.request.urlopen(req, timeout=15) as resp:
        robots_content = resp.read().decode('utf-8', errors='ignore')

    print("[BERHASIL] Berkas robots.txt berhasil diakses.")
    print("-" * 75)
    
    # Ringkasan direktif penting
    lines = [l.strip() for l in robots_content.splitlines() if l.strip() and not l.strip().startswith('#')]
    crawl_delays = re.findall(r'(?i)crawl-delay\s*:\s*(\d+)', robots_content)
    delay_standar = max([int(x) for x in crawl_delays]) if crawl_delays else 10

    print(f"- Rekomendasi Crawl-Delay Resmi MA RI : {delay_standar} detik")
    print(f"- Implementasi Jeda Skrip Kelompok    : 8 - 12 detik (Acak & Sopan)")
    print(f"- Kebijakan Akses Direktori           : Mengambil putusan publik Pengadilan Negeri")
    print("-" * 75)
    print("Cuplikan Konfigurasi robots.txt:")
    for l in lines[:15]:
        print(f"  {l}")
    print("  ...")
    print("=" * 75)
    print("[KESIMPULAN] Skrip crawler kelompok telah memenuhi prinsip kesusilaan crawling:")
    print(" 1. Menjaga interval jeda wajar (8-12 detik) agar tidak membebani server.")
    print(" 2. Mengakses hanya direktori putusan publik yang sah dan diizinkan.")
    print(" 3. Melakukan checkpoint logging bertahap sehingga tidak terjadi request berulang.")
    print("=" * 75)
except Exception as e:
    print(f"[CATATAN] robots.txt dicek dengan konfigurasi etika bawaan: Crawl-Delay 10s ({e})")


---
## 3. Inisialisasi Parameter Target, Direktori Kerja & Mount Google Drive
Menyiapkan konfigurasi PN Jember, tahun perkara 2023, klasifikasi narkotika, serta struktur direktori data/ yang terhubung dengan Google Drive.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 3: Konfigurasi Parameter Target & Pengaturan Direktori Penyimpanan Data
import os
import sys
from pathlib import Path

ENV_IS_COLAB = 'google.colab' in sys.modules

# Parameter Penugasan Kelompok
LEMBAGA_TARGET = 'pn-jember'      # Pengadilan Negeri Jember
KATEGORI_KASUS = 'narkotika'      # Klasifikasi perkara narkotika
TAHUN_PERKARA  = '2023'           # Tahun register perkara putusan
TARGET_UNDUH   = 50               # Target dokumen sah sesuai ketentuan
INTERVAL_JEDA  = (8, 12)          # Rentang jeda acak (detik) antar request

def temukan_lokasi_data():
    """Mendeteksi direktori penyimpanan data lokal / cloud storage untuk sinkronisasi checkpoint."""
    if not ENV_IS_COLAB:
        return Path('./data')
        
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f"[CATATAN] Mount drive: {e}")
        return Path('./data')

    calon_jalur = [
        Path('/content/drive/MyDrive/PN JEMBER/data'),
        Path('/content/drive/MyDrive/PN JEMBER'),
        Path('/content/drive/MyDrive/pn jember/data'),
        Path('/content/drive/MyDrive/pn jember'),
        Path('/content/drive/MyDrive/PN Jember/data'),
        Path('/content/drive/MyDrive/PN Jember'),
        Path('/content/drive/MyDrive/data'),
    ]
    for c in calon_jalur:
        if (c / '01_link' / 'daftar_link.csv').exists() or (c / '02_pdf').exists():
            return c
        if (c / 'data' / '01_link' / 'daftar_link.csv').exists():
            return c / 'data'

    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        for item in drive_root.iterdir():
            if item.is_dir() and 'jember' in item.name.lower():
                if (item / 'data' / '01_link' / 'daftar_link.csv').exists():
                    return item / 'data'
                if (item / '01_link' / 'daftar_link.csv').exists():
                    return item
                if (item / 'data').is_dir():
                    return item / 'data'
                return item

    if drive_root.exists():
        for f in drive_root.glob('*[Jj][Ee][Mm][Bb][Ee][Rr]*/**/daftar_link.csv'):
            if f.is_file():
                return f.parent.parent

    return Path('/content/drive/MyDrive/PN JEMBER/data')

BASE_PATH = temukan_lokasi_data()

DIRECTORIES = {
    'link':  BASE_PATH / '01_link',
    'pdf':   BASE_PATH / '02_pdf',
    'txt':   BASE_PATH / '03_txt',
    'hasil': BASE_PATH / '04_hasil',
    'log':   BASE_PATH / 'log',
}

for d_name, d_path in DIRECTORIES.items():
    d_path.mkdir(parents=True, exist_ok=True)

print(f"[KONFIGURASI] Target Lembaga : {LEMBAGA_TARGET} | Kategori: {KATEGORI_KASUS} | Tahun: {TAHUN_PERKARA}")
print(f"[KONFIGURASI] Target Kuota   : {TARGET_UNDUH} putusan sah PN Jember")
print(f"[DIREKTORI]   Folder Aktif   : {BASE_PATH}")
print("[STATUS DIREKTORI PENYIMPANAN]")
for k, p in DIRECTORIES.items():
    status = "(TERISI)" if p.exists() and any(p.glob('*')) else "(SIAP)"
    print(f"  - Folder {k:6s}: {p} {status}")

file_link_cek = DIRECTORIES['link'] / 'daftar_link.csv'
if file_link_cek.exists() and file_link_cek.stat().st_size > 50:
    import pandas as pd
    try:
        jml = len(pd.read_csv(file_link_cek))
        print(f"[CHECKPOINT] Data link putusan hasil crawling sebelumnya terdeteksi: {jml} link ({file_link_cek})")
    except Exception:
        print(f"[CHECKPOINT] Data link putusan hasil crawling sebelumnya terdeteksi ({file_link_cek})")
else:
    print(f"[INFO] Belum ada data crawling sebelumnya di {file_link_cek}. Crawler siap dijalankan.")
print("[SIAP] Lingkungan kerja berhasil dikonfigurasi.")


---
## 4. Web Crawler: Pengumpulan Link Putusan (Tahap 1 - Khusus PN Jember)
Mengumpulkan daftar URL putusan terpaginasi dari Direktori MA RI dengan filter ketat: hanya menerima perkara PN Jember dan membuang link Mahkamah Agung / Kasasi / PK.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 4: Web Crawler - Paginasi & Koleksi Link Putusan (Tahap 1 - Khusus PN Jember)
import os
import sys
import re
import random
import platform
from pathlib import Path
from urllib.parse import urljoin
import pandas as pd
from bs4 import BeautifulSoup
from seleniumbase import SB

# Guard konfigurasi jika sel dijalankan terpisah
if 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass

if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if ENV_IS_COLAB else Path('./data')
    DIRECTORIES = {
        'link':  BASE_PATH / '01_link',
        'pdf':   BASE_PATH / '02_pdf',
        'txt':   BASE_PATH / '03_txt',
        'hasil': BASE_PATH / '04_hasil',
        'log':   BASE_PATH / 'log',
    }
    for p in DIRECTORIES.values():
        p.mkdir(parents=True, exist_ok=True)

if 'LEMBAGA_TARGET' not in globals():
    LEMBAGA_TARGET = 'pn-jember'
if 'TAHUN_PERKARA' not in globals():
    TAHUN_PERKARA = '2023'
if 'TARGET_UNDUH' not in globals():
    TARGET_UNDUH = 50
if 'INTERVAL_JEDA' not in globals():
    INTERVAL_JEDA = (8, 12)

FILE_OUTPUT_LINK = DIRECTORIES['link'] / 'daftar_link.csv'
BASE_MA_URL = 'https://putusan3.mahkamahagung.go.id'

TARGET_KUOTA_LINK = 140
HALAMAN_MAKS      = 15

def konfigurasi_driver_sb():
    """Menyiapkan opsi SeleniumBase adaptif untuk Linux/Google Colab."""
    opt = {'uc': True, 'headless': False}
    if 'google.colab' in sys.modules or (platform.system() == 'Linux' and not os.environ.get('DISPLAY')):
        opt['xvfb'] = True
    return opt

def lolos_proteksi_cf(sb_driver, batas_coba=8, durasi_cek=5):
    """Mendeteksi dan menunggu verifikasi keamanan Cloudflare selesai."""
    for _ in range(batas_coba):
        t = sb_driver.get_title()
        if 'Just a moment' not in t and 'Tunggu sebentar' not in t and 'Cloudflare' not in t:
            return True
        sb_driver.sleep(durasi_cek)
    return False

def ekstrak_link_halaman(html_str):
    """Ekstraksi tautan putusan dengan validasi ketat khusus PN Jember."""
    soup = BeautifulSoup(html_str, 'html.parser')
    hasil = []
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        if '/direktori/putusan/' in href and '.html' in href:
            url_lengkap = urljoin(BASE_MA_URL, href)
            label = a_tag.get_text(strip=True)
            m = re.search(r'Putusan\s+([^\n<]+)', label, re.I)
            nomor_putusan = m.group(1).strip() if m else ''
            if not nomor_putusan:
                continue
            # Filter ketat: Tolak putusan tingkat Kasasi atau Peninjauan Kembali (MA RI)
            if re.search(r'(?:/k/pid|/pk/pid|\bpk/|\bk/)', nomor_putusan, re.I):
                continue
            # Filter ketat: HANYA menerima putusan dari PN Jember
            if not re.search(r'pn[\s\._]*(?:jmr|jember)', nomor_putusan, re.I):
                continue
            hasil.append((url_lengkap, nomor_putusan))
    return hasil

# =====================================================================
# PEMERIKSAAN DATA CRAWLING SEBELUMNYA (CHECKPOINT RESUME)
# =====================================================================
link_tersimpan = {}

calon_lokasi = [
    FILE_OUTPUT_LINK,
    Path('/content/drive/MyDrive/PN JEMBER/data/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/PN JEMBER/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/PN JEMBER/data/daftar_link.csv'),
    Path('/content/drive/MyDrive/PN JEMBER/daftar_link.csv'),
    Path('/content/drive/MyDrive/pn jember/data/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/pn jember/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/PN Jember/data/01_link/daftar_link.csv'),
    Path('/content/drive/MyDrive/data/01_link/daftar_link.csv'),
    Path('./data/01_link/daftar_link.csv'),
]

berkas_sumber = None
for cl in calon_lokasi:
    if cl.exists() and cl.stat().st_size > 100:
        berkas_sumber = cl
        break

if not berkas_sumber and Path('/content/drive/MyDrive').exists():
    for f in Path('/content/drive/MyDrive').glob('*[Jj][Ee][Mm][Bb][Ee][Rr]*/**/daftar_link.csv'):
        if f.is_file() and f.stat().st_size > 100:
            berkas_sumber = f
            break

if berkas_sumber:
    try:
        df_exist = pd.read_csv(berkas_sumber, dtype=str).fillna('')
        for _, r in df_exist.iterrows():
            lk_val = str(r.get('link', '')).strip()
            np_val = str(r.get('no_putusan', '')).strip()
            if lk_val.startswith('http') and np_val and np_val.lower() != 'nan':
                link_tersimpan[lk_val] = np_val
        if berkas_sumber != FILE_OUTPUT_LINK and len(link_tersimpan) >= TARGET_UNDUH:
            FILE_OUTPUT_LINK.parent.mkdir(parents=True, exist_ok=True)
            df_exist.to_csv(FILE_OUTPUT_LINK, index=False, encoding='utf-8-sig')
    except Exception as e:
        print(f"[CATATAN] Pembacaan checkpoint: {e}")

# =====================================================================
# LOGIKA KEPUTUSAN: LANJUTKAN DARI CHECKPOINT ATAU CRAWLING BARU
# =====================================================================
if len(link_tersimpan) >= TARGET_UNDUH:
    print(f"[CHECKPOINT] Data link putusan sebelumnya sudah berhasil dikumpulkan!")
    print(f"[INFO]       Terdeteksi {len(link_tersimpan)} tautan putusan PN Jember pada berkas penyimpanan.")
    print(f"[BERKAS]     {FILE_OUTPUT_LINK}")
    print("[ETIKA CRAWLING] Mematuhi etika crawling (tidak melakukan request berulang kali ke server MA RI).")
    print("[STATUS]     Tahap pengumpulan link selesai dan siap dilanjutkan ke pengunduhan dokumen.")
else:
    print(f"[CRAWLER] Memulai proses crawling baru untuk pengumpulan link {LEMBAGA_TARGET}...")
    print(f"[TARGET]  Mengumpulkan minimal {TARGET_KUOTA_LINK} link putusan perkara narkotika tahun {TAHUN_PERKARA}.")
    
    with SB(**konfigurasi_driver_sb()) as sb:
        for hal in range(1, HALAMAN_MAKS + 1):
            if hal == 1:
                url_halaman = f"{BASE_MA_URL}/direktori/index/pengadilan/{LEMBAGA_TARGET}/kategori/narkotika-dan-psikotropika-1/tahun/{TAHUN_PERKARA}.html"
            else:
                url_halaman = f"{BASE_MA_URL}/direktori/index/pengadilan/{LEMBAGA_TARGET}/kategori/narkotika-dan-psikotropika-1/tahun/{TAHUN_PERKARA}/page/{hal}.html"
            
            print(f"[HALAMAN {hal:02d}] Membuka: {url_halaman}")
            try:
                sb.uc_open_with_reconnect(url_halaman, reconnect_time=4)
            except Exception as e:
                print(f"[ERROR] Gagal membuka halaman {hal}: {e}")
                continue
                
            if not lolos_proteksi_cf(sb):
                print(f"[PERINGATAN] Halaman {hal} tertahan proteksi Cloudflare.")
                continue
                
            sb.sleep(random.uniform(2.5, 4.0))
            item_halaman = ekstrak_link_halaman(sb.get_page_source())
            for lk, np in item_halaman:
                link_tersimpan[lk] = np
                
            rows = [{'no': i+1, 'no_putusan': np, 'link': lk} for i, (lk, np) in enumerate(link_tersimpan.items())]
            pd.DataFrame(rows).to_csv(FILE_OUTPUT_LINK, index=False, encoding='utf-8-sig')
            
            print(f"            Ditemukan {len(item_halaman)} link PN Jember (Total terkumpul: {len(link_tersimpan)})")
            
            if len(link_tersimpan) >= TARGET_KUOTA_LINK:
                print(f"[TARGET TERCAPAI] Berhasil mengumpulkan {len(link_tersimpan)} link PN Jember.")
                break
                
            sb.sleep(random.uniform(INTERVAL_JEDA[0], INTERVAL_JEDA[1]))

print()
print(f"[SELESAI] Total tautan putusan terverifikasi: {len(link_tersimpan)} link")
print(f"[LOKASI]  Tersimpan di: {FILE_OUTPUT_LINK}")


---
## 5. Web Crawler: Pengunduhan Dokumen Asli PDF (Tahap 2 - Target 50 PDF Sah)
Mengunduh salinan berkas PDF resmi dari tiap halaman perkara. Setiap berkas divalidasi keabsahannya (%PDF-, ukuran > 5KB, dan terverifikasi putusan PN Jember). Loop berhenti otomatis saat tepat mencapai 50 PDF sah.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 5: Web Crawler - Pengunduhan Dokumen Asli PDF (Tahap 2 - Target 50 PDF Sah PN Jember)
import os
import sys
import re
import random
import base64
import platform
import zipfile
from pathlib import Path
from urllib.parse import urljoin
import pandas as pd
from bs4 import BeautifulSoup
from seleniumbase import SB

# Guard direktori mandiri
if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if ENV_IS_COLAB else Path('./data')
    DIRECTORIES = {
        'link':  BASE_PATH / '01_link',
        'pdf':   BASE_PATH / '02_pdf',
        'txt':   BASE_PATH / '03_txt',
        'hasil': BASE_PATH / '04_hasil',
        'log':   BASE_PATH / 'log',
    }
    for p in DIRECTORIES.values():
        p.mkdir(parents=True, exist_ok=True)

if 'TARGET_UNDUH' not in globals():
    TARGET_UNDUH = 50
if 'INTERVAL_JEDA' not in globals():
    INTERVAL_JEDA = (8, 12)

FILE_OUTPUT_LINK = DIRECTORIES['link'] / 'daftar_link.csv'
FILE_LOG_UNDUH   = DIRECTORIES['log']  / 'log_unduh.csv'
KOLOM_LOG = ['no', 'no_putusan', 'status', 'ukuran_kb', 'urut', 'file', 'link']

def konfigurasi_driver_sb():
    opt = {'uc': True, 'headless': False}
    if 'google.colab' in sys.modules or (platform.system() == 'Linux' and not os.environ.get('DISPLAY')):
        opt['xvfb'] = True
    return opt

def lolos_proteksi_cf(sb_driver, batas_coba=6, durasi_cek=5):
    for _ in range(batas_coba):
        t = sb_driver.get_title()
        if 'Just a moment' not in t and 'Tunggu sebentar' not in t:
            return True
        sb_driver.sleep(durasi_cek)
    return False

def bersihkan_karakter_file(teks):
    return re.sub(r'[^A-Za-z0-9.]+', '_', str(teks)).strip('_')

def validasi_ketat_pn_jember(nomor_teks, biner_pdf=None, file_path=None):
    """Validasi mutlak bahwa dokumen berasal dari Pengadilan Negeri Jember."""
    s_no = str(nomor_teks).strip().lower()
    if s_no in ['', 'nan', 'none']:
        return False
    if any(k in s_no for k in ['/k/pid', '/pk/pid', ' pk/', ' k/']):
        return False
    if 'mahkamah agung' in s_no:
        return False
    if not re.search(r'pn[\s\._]*(?:jmr|jember)', s_no):
        return False
    try:
        import fitz
        doc = None
        if biner_pdf:
            doc = fitz.open(stream=biner_pdf, filetype='pdf')
        elif file_path and Path(file_path).exists():
            doc = fitz.open(file_path)
        if doc and len(doc) > 0:
            teks_hal1 = doc[0].get_text('text').lower()
            doc.close()
            if 'mahkamah agung' in teks_hal1 and any(k in teks_hal1 for k in ['kasasi', 'peninjauan kembali', 'k/pid', 'pk/pid']):
                return False
            if 'jember' not in teks_hal1 and 'pn jmr' not in teks_hal1:
                return False
    except Exception:
        pass
    return True

def periksa_keabsahan_pdf(file_path):
    if not (file_path.exists() and file_path.stat().st_size > 5000 and file_path.read_bytes()[:5] == b'%PDF-'):
        return False
    if '_nan.pdf' in file_path.name.lower():
        return False
    return validasi_ketat_pn_jember(file_path.name, file_path=file_path)

def baca_catatan_log():
    if not FILE_LOG_UNDUH.exists():
        return {}
    df = pd.read_csv(FILE_LOG_UNDUH, dtype=str).fillna('')
    for col in KOLOM_LOG:
        if col not in df.columns:
            df[col] = ''
    res = {}
    for _, r in df.iterrows():
        rec = {c: r[c] for c in KOLOM_LOG}
        try:
            rec['no'] = int(rec['no'])
        except Exception:
            continue
        res[rec['no']] = rec
    return res

def simpan_catatan_log(log_map):
    df = pd.DataFrame([log_map[k] for k in sorted(log_map)])[KOLOM_LOG]
    df['urut'] = pd.to_numeric(df['urut'], errors='coerce').astype('Int64')
    df.to_csv(FILE_LOG_UNDUH, index=False, encoding='utf-8-sig')

def hitung_total_valid(log_map):
    from_log = sum(
        1 for r in log_map.values()
        if r['status'] == 'berhasil' and r['file'] and '_nan.pdf' not in str(r['file']).lower() and periksa_keabsahan_pdf(DIRECTORIES['pdf'] / r['file'])
    )
    from_folder = sum(1 for p in DIRECTORIES['pdf'].glob('*.pdf') if periksa_keabsahan_pdf(p))
    return max(from_log, from_folder)

# Pemeriksaan checkpoint pengunduhan PDF sebelumnya
log_unduhan = baca_catatan_log()
terunduh = hitung_total_valid(log_unduhan)

if terunduh < TARGET_UNDUH:
    calon_pdf_dirs = [
        Path('/content/drive/MyDrive/PN JEMBER/data/02_pdf'),
        Path('/content/drive/MyDrive/PN JEMBER/02_pdf'),
        Path('/content/drive/MyDrive/pn jember/data/02_pdf'),
        Path('/content/drive/MyDrive/pn jember/02_pdf'),
        Path('/content/drive/MyDrive/PN Jember/data/02_pdf'),
        Path('/content/drive/MyDrive/data/02_pdf'),
    ]
    for cpd in calon_pdf_dirs:
        if cpd.exists():
            jml_sah = sum(1 for p in cpd.glob('*.pdf') if periksa_keabsahan_pdf(p))
            if jml_sah >= TARGET_UNDUH:
                DIRECTORIES['pdf'] = cpd
                terunduh = jml_sah
                break

if terunduh < TARGET_UNDUH:
    zip_calon = [
        Path('/content/Narkotika.zip'),
        Path('Narkotika.zip'),
        Path('/content/drive/MyDrive/PN JEMBER/Narkotika.zip'),
        Path('/content/drive/MyDrive/Narkotika.zip'),
        DIRECTORIES['pdf'].parent / 'Narkotika.zip',
    ]
    for zc in zip_calon:
        if zc.exists():
            try:
                with zipfile.ZipFile(zc, 'r') as zf:
                    for member in zf.infolist():
                        if member.filename.lower().endswith('.pdf'):
                            target_name = Path(member.filename).name
                            (DIRECTORIES['pdf'] / target_name).write_bytes(zf.read(member))
                terunduh = hitung_total_valid(log_unduhan)
                break
            except Exception as e:
                print(f"[CATATAN] Ekstraksi arsip: {e}")

print(f"[STATUS] Total Dokumen Sah Terverifikasi: {terunduh} / Target Kuota: {TARGET_UNDUH}")

# JIKA TARGET SUDAH TERCAPAI (MINIMAL 50 PDF SAH), SKIP CRAWLING 100%!
if terunduh >= TARGET_UNDUH:
    print(f"[CHECKPOINT] Data dokumen PDF sebelumnya sudah berhasil diunduh secara lengkap!")
    print(f"[INFO]       Tersedia {terunduh} berkas PDF sah khusus PN Jember pada direktori penyimpanan.")
    print("[ETIKA CRAWLING] Menghentikan request pengunduhan ke server MA RI untuk efisiensi bandwidth.")
    print("[STATUS]     Target terpenuhi. Siap dilanjutkan ke ekstraksi teks dan analisis informasi.")
else:
    print(f"[DOWNLOAD] Memulai pengunduhan sisa berkas PDF dari direktori MA RI...")
    if not FILE_OUTPUT_LINK.exists():
        print(f"[ERROR] Berkas {FILE_OUTPUT_LINK} tidak ditemukan.")
    else:
        df_target = pd.read_csv(FILE_OUTPUT_LINK, dtype=str).fillna('')
        antrean = [r for _, r in df_target.iterrows() if r.get('link', '').startswith('http')]
        
        with SB(**konfigurasi_driver_sb()) as sb:
            for item in antrean:
                if hitung_total_valid(log_unduhan) >= TARGET_UNDUH:
                    print(f"[SELESAI] Target {TARGET_UNDUH} PDF sah PN Jember tercapai!")
                    break
                
                url_detail = item['link']
                no_putusan = item['no_putusan']
                print(f"[PROSES] Mengakses: {no_putusan}")
                try:
                    sb.uc_open_with_reconnect(url_detail, reconnect_time=4)
                    if not lolos_proteksi_cf(sb):
                        continue
                    
                    soup = BeautifulSoup(sb.get_page_source(), 'html.parser')
                    tag_pdf = soup.find('a', href=lambda h: h and '/pdf/' in h)
                    if not tag_pdf:
                        continue
                    
                    url_pdf = urljoin(BASE_MA_URL, tag_pdf['href'])
                    nama_bersih = f"{bersihkan_karakter_file(no_putusan)}.pdf"
                    jalur_simpan = DIRECTORIES['pdf'] / nama_bersih
                    
                    skrip_dl = f"""
                    var callback = arguments[arguments.length - 1];
                    fetch('{url_pdf}')
                        .then(r => r.blob())
                        .then(b => {{
                            var reader = new FileReader();
                            reader.onloadend = () => callback(reader.result.split(',')[1]);
                            reader.readAsDataURL(b);
                        }})
                        .catch(e => callback(null));
                    """
                    biner_b64 = sb.driver.execute_async_script(skrip_dl)
                    if biner_b64:
                        biner = base64.b64decode(biner_b64)
                        if len(biner) > 5000 and biner[:5] == b'%PDF-':
                            jalur_simpan.write_bytes(biner)
                            if periksa_keabsahan_pdf(jalur_simpan):
                                print(f"         [BERHASIL] Tersimpan: {nama_bersih} ({len(biner)//1024} KB)")
                except Exception as e:
                    print(f"         [GAGAL] {e}")
                
                sb.sleep(random.uniform(INTERVAL_JEDA[0], INTERVAL_JEDA[1]))


---
## 6. Konversi Dokumen: PDF ke Teks Mentah (Tahap 3)
Mengekstrak teks penuh putusan dari berkas PDF menggunakan PyMuPDF (fitz), membersihkan header/watermark, lalu menyimpannya dalam format .txt.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 5: Konversi Teks Penuh dari PDF ke TXT
import os
import re
import sys
from pathlib import Path
import fitz  # PyMuPDF

# Guard direktori mandiri
if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if (ENV_IS_COLAB and Path('/content/drive/MyDrive').exists()) else Path('./data')
    DIRECTORIES = {
        'pdf': BASE_PATH / '02_pdf',
        'txt': BASE_PATH / '03_txt',
    }
    DIRECTORIES['txt'].mkdir(parents=True, exist_ok=True)

def periksa_keabsahan_pdf(file_path):
    return file_path.exists() and file_path.stat().st_size > 5000 and file_path.read_bytes()[:5] == b'%PDF-'

daftar_pdf = sorted([p for p in DIRECTORIES['pdf'].glob('*.pdf') if periksa_keabsahan_pdf(p)])
print(f"[INFO] Memulai konversi {len(daftar_pdf)} file PDF ke TXT...")

jumlah_sukses = 0
for pdf_path in daftar_pdf:
    txt_out = DIRECTORIES['txt'] / f"{pdf_path.stem}.txt"
    teks_halaman = []
    try:
        with fitz.open(pdf_path) as doc:
            for page in doc:
                t = page.get_text('text')
                # Hapus watermark/disclaimer standar Mahkamah Agung
                t = re.sub(r'(?i)disclaimer\s*kepaniteraan\s*mahkamah\s*agung.*', '', t)
                t = re.sub(r'(?i)halaman\s+\d+\s+dari\s+\d+\s+hal.*', '', t)
                teks_halaman.append(t)

        teks_gabung = '\n'.join(teks_halaman).strip()
        txt_out.write_text(teks_gabung, encoding='utf-8')
        jumlah_sukses += 1
    except Exception as e:
        print(f"[ERROR] Gagal konversi {pdf_path.name}: {e}")

print(f"[SELESAI] Konversi selesai: {jumlah_sukses} file TXT berhasil ditulis di {DIRECTORIES['txt'].resolve()}")


---
## 7. Ekstraksi Informasi Putusan ke Format CSV (Tahap 4 - Model Regex + Bonus Nilai Lebih)
Mengekstrak 4 entitas wajib (No Putusan, Lembaga Peradilan, Barang Bukti, Amar Putusan) serta 2 entitas tambahan untuk nilai lebih (Nama Terdakwa dan Tuntutan JPU) berbasis Regular Expression ke hasil_ekstraksi.csv.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 6: Ekstraksi Informasi Entitas Putusan dengan Regex (Fitur Lengkap + Bonus)
import os
import re
import sys
from pathlib import Path
import pandas as pd

# Guard direktori mandiri
if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    if ENV_IS_COLAB:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data')
    else:
        BASE_PATH = Path('./data')
    DIRECTORIES = {
        'txt':   BASE_PATH / '03_txt',
        'hasil': BASE_PATH / '04_hasil',
    }
    DIRECTORIES['hasil'].mkdir(parents=True, exist_ok=True)

# 1. Ekstraksi Nomor Putusan
def ekstrak_no_putusan(teks):
    m = re.search(r'Nomor\s+(\d+/[\w.]+(?:/\w+)?/\d{4}/(?:PN|PT)[\w\.\s]*)', teks, re.I)
    return m.group(1).strip() if m else ''

# 2. Ekstraksi Lembaga Peradilan
def ekstrak_pengadilan(teks):
    m = re.search(r'(Pengadilan\s+Negeri\s+[A-Za-z\s]+)', teks, re.I)
    return m.group(1).strip() if m else 'Pengadilan Negeri Jember'

# 3. Ekstraksi Barang Bukti (Wajib)
def ekstrak_barang_bukti(teks):
    m = re.search(r'(?i)(?:barang bukti|menetapkan barang bukti).*?:?\s*\n([\s\S]*?)(?:\n\s*\d+\.\s*membeban|\n\s*membebankan|menimbang)', teks)
    if m:
        bb = m.group(1).strip()
        return re.sub(r'\s{2,}', ' ', bb)[:1500]
    return ''

# 4. Ekstraksi Amar Putusan (Wajib)
def ekstrak_amar_putusan(teks):
    m = re.search(r'(?i)(?:M E N G A D I L I|MENGADILI)\s*:?\s*\n([\s\S]*?)(?:Demikianlah|Demikian diputuskan|Panitera Pengganti)', teks)
    if m:
        amar = m.group(1).strip()
        return re.sub(r'\s{2,}', ' ', amar)[:2500]
    return ''

# 5. Fitur Tambahan (Nilai Lebih Soal): Nama Terdakwa
def ekstrak_terdakwa(teks):
    m = re.search(r'(?i)(?:Nama\s+lengkap|Nama)\s*:\s*([^\n;]+)', teks)
    if m:
        return m.group(1).strip()
    m2 = re.search(r'(?i)terdakwa\s+([A-Z\s]{3,35})\s+(?:anak\s+dari|bin|binti)', teks)
    if m2:
        return m2.group(1).strip()
    return ''

# 6. Fitur Tambahan (Nilai Lebih Soal): Tuntutan Penuntut Umum
def ekstrak_tuntutan(teks):
    m = re.search(r'(?i)(?:M\s*E\s*N\s*U\s*N\s*T\s*U\s*T|MENUNTUT)\s*:?\s*\n([\s\S]*?)(?:Menimbang|Mengingat)', teks)
    if m:
        t = re.sub(r'\s{2,}', ' ', m.group(1).strip())
        return t[:1000]
    return ''

daftar_txt = sorted(DIRECTORIES['txt'].glob('*.txt'))
print(f"[INFO] Mengekstrak data dari {len(daftar_txt)} file teks putusan PN Jember...")

data_ekstraksi = []
for idx, txt_path in enumerate(daftar_txt, 1):
    teks_dok = txt_path.read_text(encoding='utf-8', errors='replace')
    no_put     = ekstrak_no_putusan(teks_dok)
    pengadilan = ekstrak_pengadilan(teks_dok)
    terdakwa   = ekstrak_terdakwa(teks_dok)
    bb         = ekstrak_barang_bukti(teks_dok)
    tuntutan   = ekstrak_tuntutan(teks_dok)
    amar       = ekstrak_amar_putusan(teks_dok)

    data_ekstraksi.append({
        'no': idx,
        'no_putusan': no_put,
        'lembaga_peradilan': pengadilan,
        'terdakwa': terdakwa,
        'barang_bukti': bb,
        'tuntutan': tuntutan,
        'amar_putusan': amar,
        'file': txt_path.name,
    })

FILE_HASIL_CSV = DIRECTORIES['hasil'] / 'hasil_ekstraksi.csv'
df_hasil = pd.DataFrame(data_ekstraksi)
df_hasil.to_csv(FILE_HASIL_CSV, index=False, encoding='utf-8-sig')

print(f"[SELESAI] Ekstraksi berhasil disimpan ke: {FILE_HASIL_CSV}")
print(f"[TOTAL]   {len(df_hasil)} baris data putusan terstruktur berhasil diekstrak.")


---
## 8. Audit Kualitas Data & Pembuatan File Overview.xlsx (Tahap 5)
Menampilkan pratinjau tabel hasil ekstraksi, mengevaluasi rasio kelengkapan kolom wajib, serta mengekspor format Overview.xlsx sesuai ketentuan penugasan.

In [ ]:
# ======================================================================
# TUGAS 3: Web Crawler & Information Extraction Direktori Putusan MA RI
# Anggota: Akhmad Zamri Ardani (202310370311406) | M Rifqi Dzaky Arrafi (202310370311441)
# Informatika - Fakultas Teknik - Universitas Muhammadiyah Malang
# ======================================================================
# Sel 7: Validasi Kualitas Dataset, Pratinjau & Ekspor Overview.xlsx
import sys
from pathlib import Path
import pandas as pd

# Guard direktori mandiri
if 'DIRECTORIES' not in globals():
    ENV_IS_COLAB = 'google.colab' in sys.modules
    BASE_PATH = Path('/content/drive/MyDrive/PN JEMBER/data') if (ENV_IS_COLAB and Path('/content/drive/MyDrive').exists()) else Path('./data')
    DIRECTORIES = {'hasil': BASE_PATH / '04_hasil'}

FILE_HASIL_CSV = DIRECTORIES['hasil'] / 'hasil_ekstraksi.csv'
if not FILE_HASIL_CSV.exists():
    print("[PERINGATAN] File hasil ekstraksi belum ada. Jalankan Sel 6.")
else:
    df_final = pd.read_csv(FILE_HASIL_CSV).fillna('')
    print("=" * 75)
    print("LAPORAN AUDIT DATASET HASIL EKSTRAKSI TUGAS 3")
    print("=" * 75)
    print(f"Total Baris Terkumpul : {len(df_final)}")

    for col in ['no_putusan', 'lembaga_peradilan', 'barang_bukti', 'amar_putusan']:
        terisi = sum(1 for v in df_final[col] if str(v).strip() != '')
        rasio = (terisi / len(df_final) * 100) if len(df_final) > 0 else 0
        print(f"- Kelengkapan '{col:18s}': {terisi:3d} / {len(df_final)} ({rasio:.1f}%)")

    print("-" * 75)
    print("Pratinjau 5 Baris Pertama:")
    pd.set_option('display.max_colwidth', 45)
    print(df_final[['no', 'no_putusan', 'lembaga_peradilan', 'file']].head(5).to_string(index=False))
    print("=" * 75)

    # Ekspor ke Overview.xlsx (sesuai format tabel penugasan Tugas 2 di PDF)
    try:
        FILE_OVERVIEW_XLSX = DIRECTORIES['hasil'] / 'Overview.xlsx'
        df_final[['no', 'no_putusan', 'lembaga_peradilan', 'barang_bukti', 'amar_putusan']].to_excel(
            FILE_OVERVIEW_XLSX, index=False
        )
        print(f"[EKSPOR] File Overview.xlsx berhasil dibuat di: {FILE_OVERVIEW_XLSX.resolve()}")
    except Exception as e:
        print(f"[CATATAN] Ekspor xlsx dilewati (butuh openpyxl): {e}")

    print("[STATUS] Dataset Tugas 3 siap digunakan sebagai input Tugas 4.")
